# Enterprise Policy Copilot: Stage 1 - Synthetic Data Exploration

Welcome to Stage 1 of the **Enterprise Policy Copilot (RAG Assistant)** development journey!

### 1. What This Notebook Does
This notebook introduces and inspects our **synthetic policy dataset**, which serves as the foundational text for developing our Retrieval-Augmented Generation (RAG) assistant. Here, we:
- Load the structured synthetic policy dataset from JSON.
- Inspect the data structure using `pandas`.
- Understand document metadata and hierarchical section layouts.
- Perform statistical analysis and strict data validation checks.
- Establish how these documents will feed into the next stage: **Chunking and Text Splitting**.

### 2. Why Data Exploration is Necessary Before Chunking
In a RAG system, chunking cannot be treated as a black-box operation. Text chunking decisions depend entirely on:
- **Document structure**: Are there explicit section headings, bullet points, or paragraphs?
- **Metadata availability**: What context (`doc_id`, `document_title`, `section_heading`) must accompany each chunk so the retrieval engine can accurately cite sources?
- **Content density**: How long are policy sections, and what chunk size or overlap is appropriate?
Exploring the data first prevents downstream retrieval errors, hallucinated citations, and loss of critical context.

### 3. How This Stage Contributes to the RAG Pipeline
A complete RAG architecture consists of:
1. **Document Ingestion & Exploration** *(This Stage)*
2. **Text Chunking** (splitting policies into retrievable passages with metadata)
3. **Embeddings & Vector Indexing** (generating vector representations and building a search index)
4. **Retrieval & Reranking** (finding the most relevant chunks for user queries)
5. **Contextual Generation** (synthesizing clear answers with strict citations)

By validating our dataset structure today, we ensure our upcoming chunking module receives clean, uniform, and trustworthy inputs.


## Section 2: Import Libraries

We use Python standard library utilities and popular data analysis tools:
- **`pandas`**: High-performance data manipulation and analysis library used to represent our dataset as a tabular DataFrame.
- **`json`**: Built-in library to parse and load JSON files.
- **`pathlib.Path`**: Object-oriented filesystem path management ensuring our code works across Windows, macOS, and Linux without hardcoding path separators.


In [1]:
# Section 2: Imports
import json
from pathlib import Path
import pandas as pd

# Configure pandas display options for clear readability
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")
print(f"pandas version: {pd.__version__}")


Libraries imported successfully!
pandas version: 3.1.1


## Section 3: Load Synthetic Dataset

We locate the synthetic policies file using a **project-relative path**. 
Whether this notebook is executed from the repository root or inside the `notebooks/` directory, `pathlib.Path` dynamically resolves the correct path to `data/sample/synthetic_policies.json`.

We wrap the file loading in a `try-except` block to cleanly catch:
- `FileNotFoundError`: If the dataset file is missing.
- `json.JSONDecodeError`: If the JSON content is malformed.


In [2]:
# Section 3: Locate and load the JSON dataset using relative paths

# Dynamically resolve project root: check current working directory or parent if running inside notebooks/
current_dir = Path.cwd()
if (current_dir / "data" / "sample" / "synthetic_policies.json").exists():
    dataset_path = current_dir / "data" / "sample" / "synthetic_policies.json"
elif (current_dir.parent / "data" / "sample" / "synthetic_policies.json").exists():
    dataset_path = current_dir.parent / "data" / "sample" / "synthetic_policies.json"
else:
    dataset_path = Path("data/sample/synthetic_policies.json")

print(f"Attempting to load dataset from: {dataset_path}")

try:
    with open(dataset_path, "r", encoding="utf-8") as file:
        raw_policies_data = json.load(file)
    print(f"SUCCESS: Loaded {len(raw_policies_data)} synthetic policy records from {dataset_path.name}!")
except FileNotFoundError:
    print(f"ERROR: Dataset file not found at '{dataset_path}'. Please ensure the file exists.")
    raw_policies_data = []
except json.JSONDecodeError as err:
    print(f"ERROR: Failed to parse JSON file '{dataset_path}': {err}")
    raw_policies_data = []


Attempting to load dataset from: C:\Users\sandhiya\OneDrive\Data Science Intern\Enterprise-Policy-Copilot\data\sample\synthetic_policies.json
SUCCESS: Loaded 15 synthetic policy records from synthetic_policies.json!


## Section 4: Create DataFrame

We convert the list of JSON dictionaries into a `pandas.DataFrame`. 

A **DataFrame** is a two-dimensional tabular data structure with labeled axes (rows and columns):
- **Rows**: Represent individual policy sections.
- **Columns**: Represent the metadata and text attributes (`doc_id`, `document_title`, `section_heading`, etc.).


In [3]:
# Section 4: Convert JSON records to a pandas DataFrame
df_policies = pd.DataFrame(raw_policies_data)

# Display dataset shape (number of rows and columns)
num_rows, num_cols = df_policies.shape
print(f"Dataset Shape: {num_rows} rows (sections) x {num_cols} columns (attributes)")
print(f"Available Columns: {list(df_policies.columns)}")

# Display the first 5 records in a formatted preview
print("\n--- Preview of First 5 Records ---")
display(df_policies.head(5))


Dataset Shape: 15 rows (sections) x 6 columns (attributes)
Available Columns: ['doc_id', 'document_title', 'section_heading', 'section_content', 'version', 'effective_date']

--- Preview of First 5 Records ---


,doc_id,document_title,section_heading,section_content,version,effective_date
0,POL-AL-001,Annual Leave Policy,Eligibility,All full-time employees who have completed a minimum of six months of continuous service with the organization are eligible for annual leave. Part-time employees working at least 20 hours per week become eligible after completing 12 months of continuous service. Contract employees and interns are not eligible for annual leave under this policy. Eligibility is determined based on the employee's hire date as recorded in the HR management system. Employees on a Performance Improvement Plan may have their annual leave requests reviewed by their department head before approval.,2.1,2025-01-15
1,POL-AL-001,Annual Leave Policy,Leave Entitlement,"Eligible full-time employees are entitled to 18 working days of paid annual leave per calendar year. Part-time employees receive a prorated entitlement based on their contracted hours relative to a standard 40-hour work week. Unused annual leave up to a maximum of 5 days may be carried forward to the following calendar year, but carried-forward days must be used before March 31. Any unused carried-forward leave that is not used by this deadline will lapse automatically. Employees who join mid-year receive a prorated entitlement based on the number of remaining complete months in the calendar year.",2.1,2025-01-15
2,POL-AL-001,Annual Leave Policy,Application Procedure,"Annual leave requests must be submitted through the internal Employee Self-Service portal at least 10 working days before the intended start date. Requests for leave exceeding 5 consecutive working days require a minimum of 20 working days advance notice. All leave requests must include the start date, end date, total number of days requested, and a brief reason for leave. The direct supervisor will receive an automatic notification and must respond to the request within 3 working days. If the supervisor does not respond within the stipulated timeframe, the request will be escalated to the department head for action.",2.1,2025-01-15
3,POL-AL-001,Annual Leave Policy,Blackout Periods and Restrictions,"Annual leave may not be taken during designated organizational blackout periods, which include the final two weeks of each financial quarter and the first week of January. The HR department will publish the annual blackout period calendar by December 1 of the preceding year. No more than 30 percent of a team's members may be on annual leave simultaneously. If multiple employees request the same dates, priority is given based on the date the request was submitted, followed by the employee's length of service. Exceptions to blackout period restrictions require written approval from the division director.",2.1,2025-01-15
4,POL-AL-001,Annual Leave Policy,Cancellation and Modification,"Approved annual leave may be cancelled or modified by submitting a revised request through the Employee Self-Service portal at least 5 working days before the original start date. Cancellations made less than 5 working days before the leave start date require direct approval from the supervisor and must include a written justification. Cancelled leave days are returned to the employee's available balance immediately upon cancellation approval. If the organization requires an employee to cancel approved leave due to critical business needs, the employee will receive priority consideration for future leave requests covering the same period. Repeated cancellation of approved leave by the employee may be flagged for review by the HR department.",2.1,2025-01-15


## Section 5: Understand Metadata

Every record in our dataset represents a distinct **policy section** with essential metadata fields:

| Field Name | Type | Purpose in RAG Pipeline |
|---|---|---|
| **`doc_id`** | String | Unique identifier for the parent policy document (e.g., `POL-AL-001`). Used to filter documents and manage versioning. |
| **`document_title`** | String | Human-readable document name (e.g., *Annual Leave Policy*). Used in user-facing citations. |
| **`section_heading`** | String | The title of the specific section (e.g., *Eligibility*, *Leave Entitlement*). Helps the retrieval engine match user intent. |
| **`section_content`** | String | The actual authoritative policy text that will be chunked, embedded, and retrieved. |
| **`version`** | String | Revision version of the document (e.g., `2.1`). Ensures only active policy versions are queried. |
| **`effective_date`** | String | The formal effective date (`YYYY-MM-DD`). Allows temporal reasoning (e.g., "What was the policy in 2025?"). |


In [4]:
# Section 5: Inspect a concrete example record to understand metadata fields
sample_record = df_policies.iloc[0]

print("=== Metadata Breakdown for a Sample Record ===")
print(f"1. doc_id           : {sample_record['doc_id']}")
print(f"2. document_title    : {sample_record['document_title']}")
print(f"3. section_heading   : {sample_record['section_heading']}")
print(f"4. version           : {sample_record['version']}")
print(f"5. effective_date    : {sample_record['effective_date']}")
print("6. section_content   :")
print(f"   \"{sample_record['section_content']}\"")


=== Metadata Breakdown for a Sample Record ===
1. doc_id           : POL-AL-001
2. document_title    : Annual Leave Policy
3. section_heading   : Eligibility
4. version           : 2.1
5. effective_date    : 2025-01-15
6. section_content   :
   "All full-time employees who have completed a minimum of six months of continuous service with the organization are eligible for annual leave. Part-time employees working at least 20 hours per week become eligible after completing 12 months of continuous service. Contract employees and interns are not eligible for annual leave under this policy. Eligibility is determined based on the employee's hire date as recorded in the HR management system. Employees on a Performance Improvement Plan may have their annual leave requests reviewed by their department head before approval."


## Section 6: Explore Individual Policies

Our dataset contains three distinct policy documents. Here, we group and inspect all sections belonging to each policy to verify coherence, completeness, and clarity.


In [5]:
# Section 6: Display sections for each policy document separately
unique_doc_ids = df_policies['doc_id'].unique()

for doc_id in unique_doc_ids:
    policy_df = df_policies[df_policies['doc_id'] == doc_id]
    doc_title = policy_df['document_title'].iloc[0]
    doc_ver = policy_df['version'].iloc[0]
    doc_date = policy_df['effective_date'].iloc[0]
    
    print("=" * 80)
    print(f"POLICY: {doc_title} (ID: {doc_id}) | Version: {doc_ver} | Effective: {doc_date}")
    print(f"Total Sections: {len(policy_df)}")
    print("=" * 80)
    
    for idx, (_, row) in enumerate(policy_df.iterrows(), start=1):
        print(f"\n[{idx}] Section Heading: {row['section_heading']}")
        print(f"    Content: {row['section_content']}")
    print("\n")



POLICY: Annual Leave Policy (ID: POL-AL-001) | Version: 2.1 | Effective: 2025-01-15
Total Sections: 5

[1] Section Heading: Eligibility
    Content: All full-time employees who have completed a minimum of six months of continuous service with the organization are eligible for annual leave. Part-time employees working at least 20 hours per week become eligible after completing 12 months of continuous service. Contract employees and interns are not eligible for annual leave under this policy. Eligibility is determined based on the employee's hire date as recorded in the HR management system. Employees on a Performance Improvement Plan may have their annual leave requests reviewed by their department head before approval.

[2] Section Heading: Leave Entitlement
    Content: Eligible full-time employees are entitled to 18 working days of paid annual leave per calendar year. Part-time employees receive a prorated entitlement based on their contracted hours relative to a standard 40-hour wo

## Section 7: Basic Dataset Analysis

We compute summary statistics across our dataset to assess:
- The total volume of policies and sections.
- The distribution of sections per policy.
- Data completeness (identifying any null or missing values).
- Data cleanliness (identifying any duplicate records).


In [6]:
# Section 7: Dataset Statistical Analysis

total_docs = df_policies['doc_id'].nunique()
total_sections = len(df_policies)
sections_per_policy = df_policies.groupby(['doc_id', 'document_title']).size()
unique_ids = df_policies['doc_id'].unique().tolist()
missing_values = df_policies.isnull().sum()
duplicate_count = df_policies.duplicated().sum()

print("=== DATASET SUMMARY STATISTICS ===")
print(f"Total Unique Policies  : {total_docs}")
print(f"Total Policy Sections  : {total_sections}")
print(f"Unique Document IDs    : {unique_ids}")
print(f"Duplicate Records Count: {duplicate_count}")

print("\n--- Sections per Policy ---")
for (did, title), count in sections_per_policy.items():
    print(f"  * {did} ({title}): {count} sections")

print("\n--- Missing Values Count by Column ---")
for col, missing_count in missing_values.items():
    print(f"  * {col:<18} : {missing_count} missing")


=== DATASET SUMMARY STATISTICS ===
Total Unique Policies  : 3
Total Policy Sections  : 15
Unique Document IDs    : ['POL-AL-001', 'POL-SL-002', 'POL-LA-003']
Duplicate Records Count: 0

--- Sections per Policy ---
  * POL-AL-001 (Annual Leave Policy): 5 sections
  * POL-LA-003 (Leave Approval Policy): 5 sections
  * POL-SL-002 (Sick Leave Policy): 5 sections

--- Missing Values Count by Column ---
  * doc_id             : 0 missing
  * document_title     : 0 missing
  * section_heading    : 0 missing
  * section_content    : 0 missing
  * version            : 0 missing
  * effective_date     : 0 missing


## Section 8: Data Validation

Strict validation ensures data integrity before running downstream NLP and RAG pipelines.
We test:
1. **Required Columns Check**: All 6 required fields must exist.
2. **Missing Field Check**: No column may contain null or empty values.
3. **Document ID Consistency**: Each `doc_id` must map to exactly one `document_title`.
4. **Date Format Check**: All dates must adhere to the `YYYY-MM-DD` ISO format.
5. **Minimum Sections Requirement**: Each document must contain at least 4 sections.

We explicitly display **PASS** or **FAIL** for each criterion.


In [7]:
# Section 8: Strict Data Validation Routine
import re

print("=== STARTING DATA VALIDATION CHECKS ===\n")
validation_passed = True

# 1. Required columns check
expected_columns = {"doc_id", "document_title", "section_heading", "section_content", "version", "effective_date"}
actual_columns = set(df_policies.columns)
if expected_columns.issubset(actual_columns):
    print("[PASS] Check 1: All required columns exist.")
else:
    missing_cols = expected_columns - actual_columns
    print(f"[FAIL] Check 1: Missing required columns: {missing_cols}")
    validation_passed = False

# 2. No missing or null values
has_nulls = df_policies.isnull().values.any()
has_empty_strings = (df_policies.astype(str).apply(lambda x: x.str.strip()) == "").values.any()
if not has_nulls and not has_empty_strings:
    print("[PASS] Check 2: No null or empty values found in any record.")
else:
    print("[FAIL] Check 2: Dataset contains null or blank string values.")
    validation_passed = False

# 3. Document ID Consistency: 1-to-1 mapping between doc_id and document_title
id_title_mapping = df_policies.groupby('doc_id')['document_title'].nunique()
inconsistent_ids = id_title_mapping[id_title_mapping > 1]
if len(inconsistent_ids) == 0:
    print("[PASS] Check 3: Document IDs are strictly consistent with Document Titles.")
else:
    print(f"[FAIL] Check 3: Inconsistent doc_ids mapped to multiple titles: {inconsistent_ids.to_dict()}")
    validation_passed = False

# 4. Date format validation (YYYY-MM-DD)
date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
invalid_dates = df_policies[~df_policies['effective_date'].astype(str).str.match(date_pattern)]
if len(invalid_dates) == 0:
    print("[PASS] Check 4: All effective dates conform to the 'YYYY-MM-DD' format.")
else:
    print(f"[FAIL] Check 4: Found invalid dates:\n{invalid_dates[['doc_id', 'effective_date']]}")
    validation_passed = False

# 5. Minimum section requirement (>= 4 sections per policy)
min_sections = df_policies.groupby('doc_id').size()
failing_policies = min_sections[min_sections < 4]
if len(failing_policies) == 0:
    print("[PASS] Check 5: All policies have at least 4 sections (minimum requirement met).")
else:
    print(f"[FAIL] Check 5: Policies with fewer than 4 sections: {failing_policies.to_dict()}")
    validation_passed = False

# Overall validation status
print("\n" + "=" * 40)
if validation_passed:
    print("OVERALL RESULT: ALL DATA VALIDATION CHECKS PASSED [OK]")
else:
    print("OVERALL RESULT: DATA VALIDATION FAILED [ERROR]")
print("=" * 40)


=== STARTING DATA VALIDATION CHECKS ===

[PASS] Check 1: All required columns exist.
[PASS] Check 2: No null or empty values found in any record.
[PASS] Check 3: Document IDs are strictly consistent with Document Titles.
[PASS] Check 4: All effective dates conform to the 'YYYY-MM-DD' format.
[PASS] Check 5: All policies have at least 4 sections (minimum requirement met).

OVERALL RESULT: ALL DATA VALIDATION CHECKS PASSED [OK]


## Section 9: Connection to Chunking

In Stage 2, we will design **chunking strategies** to split text into optimal pieces for embedding models and vector indexing.

### Why Section Structure & Metadata Matter
When a user asks:
> *"How many days of annual leave can I carry forward to next year?"*

The retrieval system should not just return a raw snippet of text. It must return a **chunk object** containing:
1. **The chunk text**: Containing the carry-forward rules.
2. **`document_title` & `doc_id`**: Proving the rule comes from the *Annual Leave Policy* (and not Sick Leave or General Leave).
3. **`section_heading`**: Specifying it originates from *Leave Entitlement*.
4. **`effective_date` & `version`**: Confirming the policy is up to date.

### Conceptual Walkthrough: From Section to Enriched Chunk
Below is a demonstration of how a policy section's metadata will accompany each generated text chunk:


In [8]:
# Section 9: Conceptual Demonstration of Metadata Preservation for Chunking
# (Note: Chunking algorithms will be formally implemented in Stage 2)

sample_section = df_policies.iloc[1]  # 'Leave Entitlement' section of Annual Leave Policy

# Conceptual representation of how a chunk will be formatted in Stage 2
mock_chunk = {
    "chunk_id": f"{sample_section['doc_id']}_sec02_chunk01",
    "text": sample_section['section_content'],
    "metadata": {
        "doc_id": sample_section['doc_id'],
        "document_title": sample_section['document_title'],
        "section_heading": sample_section['section_heading'],
        "version": sample_section['version'],
        "effective_date": sample_section['effective_date']
    }
}

print("=== Conceptual Enriched Chunk Structure ===")
print(json.dumps(mock_chunk, indent=2))
print("\nNotice how the chunk retains full document lineage so the RAG assistant can provide precise citations!")


=== Conceptual Enriched Chunk Structure ===
{
  "chunk_id": "POL-AL-001_sec02_chunk01",
  "text": "Eligible full-time employees are entitled to 18 working days of paid annual leave per calendar year. Part-time employees receive a prorated entitlement based on their contracted hours relative to a standard 40-hour work week. Unused annual leave up to a maximum of 5 days may be carried forward to the following calendar year, but carried-forward days must be used before March 31. Any unused carried-forward leave that is not used by this deadline will lapse automatically. Employees who join mid-year receive a prorated entitlement based on the number of remaining complete months in the calendar year.",
  "metadata": {
    "doc_id": "POL-AL-001",
    "document_title": "Annual Leave Policy",
    "section_heading": "Leave Entitlement",
    "version": "2.1",
    "effective_date": "2025-01-15"
  }
}

Notice how the chunk retains full document lineage so the RAG assistant can provide precise citat

## Section 10: Summary & Next Steps

### 1. What We Created
- A structured, realistic synthetic policy dataset saved as `data/sample/synthetic_policies.json`.
- 3 distinct enterprise policies:
  1. **Annual Leave Policy** (`POL-AL-001`, 5 sections)
  2. **Sick Leave Policy** (`POL-SL-002`, 5 sections)
  3. **Leave Approval Policy** (`POL-LA-003`, 5 sections)

### 2. What We Explored
- Tabular representation using `pandas.DataFrame`.
- Complete metadata lineage (`doc_id`, `document_title`, `section_heading`, `section_content`, `version`, `effective_date`).
- Content distribution and policy-specific section breakdowns.

### 3. What We Validated
- [x] All 6 required columns are present.
- [x] Zero null or empty values across all fields.
- [x] Consistent 1-to-1 mapping between `doc_id` and `document_title`.
- [x] Dates strictly adhere to `YYYY-MM-DD`.
- [x] Every policy meets the requirement of having >= 4 meaningful sections.

### 4. What Stage 2 Will Implement
In **Stage 2: Chunking Strategies**, we will:
- Implement **Fixed-Size Chunking** with character and token boundaries.
- Implement **Overlap Strategies** to preserve context across chunk boundaries.
- Benchmark chunk sizes (e.g., 200, 500, 1000 characters) against retrieval precision.
- Build clean chunk objects with metadata inheritance ready for local embedding models.
